# Diferencias Finitas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_1.ipynb)


### Antes de empezar (solo si estás en Google Colab)

Colab arranca con un runtime de **Python** por defecto, no de Julia. Para poder correr este notebook ahí es necesario cambiar el entorno de ejecución a Julia.


---

### Setup

Paquetes usados en este notebook.

In [1]:
using LinearAlgebra, Plots

>Para instalar en Colab algun paquete se debe hacer lo siguiente:
>
>using Pkg
>
>Pkg.add("NombreDelPaquete")

### 1. Discretización de derivadas

Una forma natural de aproximar la derivada de una función $f$ en un punto $x$ es a partir de su definición como límite, reemplazando el límite por un cociente incremental con un paso $h$ finito. Según qué puntos se usen, se obtienen distintas discretizaciones:

- **Diferencias hacia adelante (forward):** $f'(x) \approx \dfrac{f(x+h)-f(x)}{h}$
- **Diferencias hacia atrás (backward):** $f'(x) \approx \dfrac{f(x)-f(x-h)}{h}$
- **Diferencias centradas:** $f'(x) \approx \dfrac{f(x+h)-f(x-h)}{2h}$

Las tres opciones se implementan en una única función, seleccionando el método por un argumento:

In [5]:
function derivada(u,x;method="forward",h=0.01)
    if method == "forward"
        return (u(x+h)-u(x))/h
    elseif method == "backward"
        return (u(x)-u(x-h))/h
    elseif method == "centradas"
        return (u(x+h)-u(x-h))/(2h)
    end
end

f(x) = sin(x)

∇f(x) = cos(x)


∇f (generic function with 1 method)

Tomando $f(x) = \sin(x)$, cuya derivada exacta es $\cos(x)$, comparamos la aproximación numérica de $f'(1)$ contra el valor exacto para distintos $h$, y estimamos el **orden de convergencia** de cada método (el exponente $p$ tal que el error se comporta como $\mathcal{O}(h^p)$).

En lugar de ajustar una recta global, estimamos el orden **localmente**, comparando cada par de valores consecutivos de $h$. Si el error se comporta como $E(h) \approx C h^p$, entonces

$$ \frac{E(h_i)}{E(h_{i+1})} \approx \left(\frac{h_i}{h_{i+1}}\right)^p \quad \Longrightarrow \quad p \approx \frac{\log\big(E(h_i)/E(h_{i+1})\big)}{\log\big(h_i/h_{i+1}\big)} $$

Esto tiene la ventaja de no depender de un ajuste global: si el orden cambiara para $h$ muy chico (por ejemplo, por efecto del redondeo de punto flotante), lo veríamos reflejado directamente en la tabla de valores locales de $p$, en vez de quedar diluido en una única pendiente promedio.

In [6]:
function orden(f,∇f,x;method="forward")
    h = [0.1,0.01,0.001,0.0001]
    error = zeros(length(h))
    for i=1:length(h)
        error[i] = abs(∇f(x) - derivada(f,x,h=h[i],method=method))
    end

    println("h\t\terror")
    for i=1:length(h)
        println(h[i], "\t", error[i])
    end
    println()

    println("Orden estimado (pares sucesivos de h):")
    for i=1:length(h)-1
        p = log(error[i]/error[i+1]) / log(h[i]/h[i+1])
        println("  p(h=", h[i], " → h=", h[i+1], ") ≈ ", round(p, digits=4))
    end

    plot(log.(h), log.(error), marker=:circle, xlabel="log(h)", ylabel="log(error)",
         label="método: "*method, legend=:topleft)
end

orden (generic function with 1 method)

**Método forward:**

In [ ]:
orden(f,∇f,1)

**Método de diferencias centradas:**

In [ ]:
#metodo centradas
orden(f,∇f,1,method="centradas")

**Conclusión:** las diferencias forward y backward tienen orden 1 (el error decrece linealmente con $h$), mientras que las diferencias centradas alcanzan orden 2 — para el mismo $h$ el error es sensiblemente menor. Esto se debe a que, al combinar $u(x+h)$ y $u(x-h)$ simétricamente, se cancela el término de orden $h$ en el desarrollo de Taylor.

### ¿Qué pasa si $h$ es demasiado chico?

Hasta ahora elegimos $h$ en un rango donde domina el **error de truncamiento** (el que viene de cortar la serie de Taylor), y por eso el error baja con el orden esperado a medida que $h \to 0$. Pero en la práctica $f(x)$ se evalúa con aritmética de punto flotante, de precisión finita: cada evaluación tiene un error de redondeo del orden de la unidad de redondeo de la máquina, $\epsilon_{\text{mach}} \approx 2.22\times10^{-16}$ para `Float64`.

Al calcular $f(x+h)-f(x)$ con $h$ muy chico, restamos dos números muy parecidos: el resultado sufre **cancelación catastrófica**, perdiendo precisión relativa, y ese error se amplifica al dividir por $h$. El error total combina entonces dos efectos que compiten:

$$ E(h) \;\approx\; \underbrace{C_1 h^p}_{\text{truncamiento}} \;+\; \underbrace{C_2\,\frac{\epsilon_{\text{mach}}}{h}}_{\text{redondeo}} $$

Para $h$ grande domina el truncamiento (el error baja al achicar $h$); para $h$ muy chico domina el redondeo (el error empieza a **subir**). Existe entonces un $h$ óptimo, ni muy grande ni muy chico, que minimiza el error total — y a partir de ahí, achicar $h$ deja de ayudar.

In [ ]:
method="forward"
x = 1
h = [10.0^k for k in -1:-1:-16]
for hi in h
    println("h = ",hi," error = ",abs(∇f(x) - derivada(f,x,h=hi,method=method)))
end

In [13]:
function error_vs_h(f,∇f,x;method="forward")
    h = [10.0^k for k in -1:-1:-16]
    error = [abs(∇f(x) - derivada(f,x,h=hi,method=method)) for hi in h]
    return h, error
end

h_f, err_f = error_vs_h(f,∇f,1,method="forward");
h_c, err_c = error_vs_h(f,∇f,1,method="centradas");

In [ ]:
plot(log10.(h_f), log10.(err_f), marker=:circle, label="forward",
     xlabel="log10(h)", ylabel="log10(error)", legend=:bottomleft)
plot!(log10.(h_c), log10.(err_c), marker=:circle, label="centradas")

In [15]:
println("h que minimiza el error (numérico):")
println("  forward:   ", h_f[argmin(err_f)])
println("  centradas: ", h_c[argmin(err_c)])
println()
println("Estimación teórica (minimizando C1*h^p + C2*eps/h):")
println("  forward   (p=1) → sqrt(eps)     ≈ ", sqrt(eps(Float64)))
println("  centradas (p=2) → eps^(1/3)     ≈ ", eps(Float64)^(1/3))

h que minimiza el error (numérico):
  forward:   1.0e-8
  centradas: 1.0e-5

Estimación teórica (minimizando C1*h^p + C2*eps/h):
  forward   (p=1) → sqrt(eps)     ≈ 1.4901161193847656e-8
  centradas (p=2) → eps^(1/3)     ≈ 6.055454452393343e-6


**Conclusión:** el gráfico deja de ser una recta para $h$ muy chico: después de bajar con la pendiente esperada (orden 1 o 2, según el método), el error alcanza un mínimo y luego **empieza a crecer**. Ese mínimo aproxima el $h$ óptimo, y coincide razonablemente con la estimación teórica. Notar además que, como la centrada tiene mayor orden, tolera achicar $h$ un poco más antes de que el redondeo tome el control — pero eventualmente el mismo fenómeno aparece en cualquier método.

**Moraleja:** "más chico" no es sinónimo de "más preciso" cuando se trabaja con aritmética de punto flotante; hay un límite práctico impuesto por la precisión de la máquina, más allá del cual seguir refinando $h$ es contraproducente.

---

### 2. Euler explícito, Euler implícito y método $\theta$

Consideramos el problema de valores iniciales

$$y'(t) = -a\,y(t), \qquad y(0)=1, \qquad a>0,$$

cuya solución exacta es $y(t) = e^{-at}$ (siempre decreciente en módulo, sin oscilar).

Para un paso $\Delta t$ fijo, definimos $y^n \approx y(t_n) = y(n\Delta t)$ mediante tres discretizaciones:

- **Euler explícito:** $\dfrac{y^{n+1}-y^n}{\Delta t} = -a y^n \;\Longrightarrow\; y^{n+1} = (1-a\Delta t)\,y^n$
- **Euler implícito:** $\dfrac{y^{n+1}-y^n}{\Delta t} = -a y^{n+1} \;\Longrightarrow\; y^{n+1} = \dfrac{y^n}{1+a\Delta t}$
- **Método $\theta=\tfrac12$ (Crank-Nicolson):** $\dfrac{y^{n+1}-y^n}{\Delta t} = -a\left(\tfrac12 y^{n+1}+\tfrac12 y^n\right) \;\Longrightarrow\; y^{n+1} = \dfrac{1-\tfrac12 a\Delta t}{1+\tfrac12 a\Delta t}\,y^n$

En los tres casos $y^{n+1} = R(a\Delta t)\,y^n$ para un **factor de amplificación** $R$ distinto según el método, y por lo tanto $y^n = R(a\Delta t)^n$. Todo el comportamiento cualitativo (decaimiento, oscilación, divergencia) queda determinado por $|R|$.

In [ ]:
function euler_explicito(a, Δt, tf; y0=1.0)
    n = Int(round(tf/Δt))
    t = [i*Δt for i in 0:n]
    y = zeros(n+1)
    y[1] = y0
    for i in 1:n
        y[i+1] = (1 - a*Δt)*y[i]
    end
    return t, y
end

function euler_implicito(a, Δt, tf; y0=1.0)
    n = Int(round(tf/Δt))
    t = [i*Δt for i in 0:n]
    y = zeros(n+1)
    y[1] = y0
    for i in 1:n
        y[i+1] = y[i] / (1 + a*Δt)
    end
    return t, y
end

function theta_metodo(a, Δt, tf; y0=1.0, θ=0.5)
    #completar
end

theta_metodo (generic function with 1 method)

Graficamos las tres discretizaciones para $a=7$, distintos $\Delta t$ y $0<t<3$, junto con la solución exacta:

In [ ]:
a = 7.0
tf = 3.0
Δts = [0.05, 0.1, 0.2, 0.25, 0.3, 0.4]

p1 = plot(t->exp(-a*t), 0, tf, label="exacta", linewidth=2, linestyle=:dash, color=:black,
          xlabel="t", ylabel="y", title="Euler explícito (a=7)", legend=:topright)

Δt = Δts[1] 
t,y = euler_explicito(a,Δt,tf)
plot!(p1, t, y, marker=:circle, markersize=2, label="Δt=$Δt")          
#for Δt in Δts
#    t, y = euler_explicito(a, Δt, tf)
#    plot!(p1, t, y, marker=:circle, markersize=2, label="Δt=$Δt")
#end
p1

In [ ]:
p2 = plot(t->exp(-a*t), 0, tf, label="exacta", linewidth=2, linestyle=:dash, color=:black,
          xlabel="t", ylabel="y", title="Euler implícito (a=7)", legend=:topright)

Δt = Δts[1] 
t,y = euler_implicito(a,Δt,tf)
plot!(p2, t, y, marker=:circle, markersize=2, label="Δt=$Δt")
#for Δt in Δts
#    t, y = euler_implicito(a, Δt, tf)
#    plot!(p2, t, y, marker=:circle, markersize=2, label="Δt=$Δt")
#end
p2

In [ ]:
#completar para el metodo theta

**Observación:** con Euler implícito y el método $\theta=1/2$, $y^n\to 0$ para **cualquier** $\Delta t>0$: son incondicionalmente estables (la solución nunca diverge, aunque para $\Delta t$ grande puede oscilar de signo en el caso $\theta=1/2$). Con Euler explícito, en cambio, se ve claramente que para $\Delta t$ grande la solución numérica deja de parecerse a la exacta: empieza a oscilar y, si $\Delta t$ es demasiado grande, diverge.

#### (a) Estimación numérica del $\Delta t$ crítico (Euler explícito)

Como $y^n = (1-a\Delta t)^n$, se tiene $|y^n| = |1-a\Delta t|^n$. Este factor decae monótonamente en módulo con $n$ si y solo si $|1-a\Delta t| < 1$, es decir

$$0 < \Delta t < \frac{2}{a}.$$

Para $\Delta t > 2/a$ el módulo $|1-a\Delta t|$ supera a 1 y $|y^n|$ crece sin cota (diverge). Notar que el signo de $y^n$ ya empieza a alternar antes, para $\Delta t>1/a$ (cuando $1-a\Delta t$ se vuelve negativo), pero el **módulo** sigue decayendo mientras $\Delta t<2/a$; recién a partir de $\Delta t=2/a$ el módulo deja de decaer monótonamente.

Hacemos una búsqueda numérica: para cada $\Delta t$ de una grilla fina, corremos Euler explícito y chequeamos si $|y^n|$ es monótonamente decreciente en $n$. Buscamos el primer $\Delta t$ para el que deja de serlo.

In [12]:
function decae_monotonamente(y; tol=1e-12)
    ay = abs.(y)
    for i in 1:length(ay)-1
        if ay[i+1] > ay[i] + tol
            return false
        end
    end
    return true
end

function Δt_critico_numerico(a, tf; Δt_min=0.01, Δt_max=0.6, paso=0.0005, y0=1.0)
    for Δt in Δt_min:paso:Δt_max
        t, y = euler_explicito(a, Δt, tf, y0=y0)
        if !decae_monotonamente(y)
            return Δt
        end
    end
    return nothing
end

Δt_critico_numerico (generic function with 1 method)

In [13]:
a = 7.0
Δt_num = Δt_critico_numerico(a, 3.0)

println("Δt crítico estimado numéricamente: ", Δt_num)
println("Cota teórica 2/a = ", 2/a)
println("Diferencia: ", abs(Δt_num - 2/a))

Δt crítico estimado numéricamente: 0.286
Cota teórica 2/a = 0.2857142857142857
Diferencia: 0.00028571428571427804


**Conclusión:** la búsqueda numérica ubica el $\Delta t$ crítico muy cerca de $2/a = 2/7 \approx 0.2857$, la cota teórica de estabilidad de Euler explícito para esta ecuación. Por debajo de ese valor $|y^n|$ decae monótonamente a 0 (con posible cambio de signo si $\Delta t>1/a$); apenas se cruza $2/a$, $|1-a\Delta t|>1$ y la solución numérica diverge en módulo, independientemente de qué tan chico haya sido el error de truncamiento local — es un problema de **estabilidad**, no de precisión. Esto es justamente lo que ya habíamos visto en la sección de discretización de derivadas con el análisis de $E(h)$: acá el fenómeno es análogo pero para la propagación del esquema en $n$, no para la aproximación puntual de una derivada.